# TFG V5 - ΔC phase + local mixer

This version implements the oracle phase is driven by the signed domain-wall signal between consecutive window costs.

## Represented problem

- `0`: free cell.
- `1`: occupied cell.
- Each index `i` represents a valid starting position for a window `M` inside the grid `N`.
- A window is valid if all its cells are `0`.

For each index, define:

```text
ΔC(i) = C(i+1) - C(i)
```

where `C(i)` is the number of occupied cells in `window_i`. Negative values identify descents into freer regions.

## Circuit flow

1. Prepare a uniform superposition over window indices.
2. Reversibly load `window_i` into the register `m`.
3. Reversibly load `window_{i+1}` into the register `m2`.
4. Compute the two costs into ancilla registers and apply the phase `exp(i·θ·ΔC(i))`.
5. Uncompute the ancilla registers, then `m2`, then `m`.
6. Apply a local mixer on `idx`, usually between geometrically neighboring windows.
7. Repeat the block several times.

## Main registers

- `n`: fixed grid.
- `idx`: index of the candidate window.
- `m`: window `i` temporarily loaded.
- `m2`: shifted window `i+1` temporarily loaded.
- `anc`: reversible cost ancilla for the signed difference oracle.

## Purpose of this version

V5 studies whether the `delta cost phase + mixer` block concentrates amplitude on descent positions and the free regions they enter. The code is modular so that the mixer, the parameters `theta`, `mixer_angle`, `repetitions`, and the 1D/2D/ND geometry can be changed easily.


In [ ]:
import os
if not os.path.exists("TFG_V5.ipynb"):
    raise RuntimeError("Run this notebook from the folder that contains TFG_V5.ipynb; output folders are intentionally local to that folder.")
os.environ.setdefault("MPLCONFIGDIR", os.path.join(os.getcwd(), ".matplotlib"))
os.environ.setdefault("XDG_CACHE_HOME", os.path.join(os.getcwd(), ".cache"))

import numpy as np
import qiskit

from math import prod
from qiskit import QuantumCircuit, QuantumRegister
from qiskit.quantum_info import Statevector
from qiskit.circuit.library import MCXGate, UnitaryGate

print(qiskit.__version__)

In [ ]:
# =========================================================
# ND geometry and classical utilities
# =========================================================

def validate_problem(N, M):
    if len(N) != len(M):
        raise ValueError("N and M must have the same dimension.")
    for d, (n_d, m_d) in enumerate(zip(N, M)):
        if n_d <= 0 or m_d <= 0:
            raise ValueError(f"N[{d}] and M[{d}] must be positive.")
        if m_d > n_d:
            raise ValueError(f"M[{d}] cannot be greater than N[{d}].")


def coord_to_index(coord, dims):
    """Converts ND coordinates to a row-major linear index.
    Example with dims = [4,4]:
    (0,0) -> 0    (0,1) -> 1    (0,2) -> 2    (0,3) -> 3
    (1,0) -> 4    (1,1) -> 5    (1,2) -> 6    (1,3) -> 7
    (2,0) -> 8    (2,1) -> 9    (2,2) -> 10   (2,3) -> 11
    (3,0) -> 12   (3,1) -> 13   (3,2) -> 14   (3,3) -> 15
    """
    idx_lin = 0
    stride = 1
    for d in reversed(range(len(dims))):
        idx_lin += coord[d] * stride
        stride *= dims[d]
    return idx_lin


def index_to_coord(index, dims):
    """Converts a row-major linear index to ND coordinates.
    Example with dims = [4,4]:
    0 -> (0,0)    1 -> (0,1)    2 -> (0,2)    3 -> (0,3)
    4 -> (1,0)    5 -> (1,1)    6 -> (1,2)    7 -> (1,3)
    8 -> (2,0)    9 -> (2,1)    10 -> (2,2)   11 -> (2,3)
    12 -> (3,0)   13 -> (3,1)    14 -> (3,2)   15 -> (3,3)
    """
    coord = [0] * len(dims)
    rem = index
    for d in reversed(range(len(dims))):
        coord[d] = rem % dims[d]
        rem //= dims[d]
    return tuple(coord)


def valid_starts_nd(N, M):
    """Valid starting coordinates for a window M inside N."""
    return list(np.ndindex(tuple(N[d] - M[d] + 1 for d in range(len(N)))))


def window_qubits_nd(start, N, M):
    """Linear indices of the cells covered by the window starting at start."""
    qubits = []
    for offset in np.ndindex(tuple(M)):
        coord = tuple(start[d] + offset[d] for d in range(len(N)))
        qubits.append(coord_to_index(coord, N))
    return qubits


def normalize_coord(coord, D):
    if D == 1 and isinstance(coord, int):
        return (coord,)
    return tuple(coord)


def build_grid_bits(N, occupied_coords):
    """Returns a classical vector with 1 on occupied cells and 0 on free cells."""
    D = len(N)
    grid = [0] * prod(N)
    for raw_coord in occupied_coords:
        coord = normalize_coord(raw_coord, D)
        if len(coord) != D:
            raise ValueError(f"Coordinate {coord} does not have dimension {D}.")
        for d, x in enumerate(coord):
            if x < 0 or x >= N[d]:
                raise ValueError(f"Coordinate {coord} is outside the grid N={N}.")
        grid[coord_to_index(coord, N)] = 1
    return grid


def compute_window_cost_classical(grid_bits, start, N, M):
    """C(i): number of ones in the window associated with start."""
    return sum(grid_bits[q] for q in window_qubits_nd(start, N, M))


def window_string_classical(grid_bits, start, N, M):
    return ''.join(str(grid_bits[q]) for q in window_qubits_nd(start, N, M))


def get_valid_indices(grid_bits, starts, N, M):
    return [i for i, start in enumerate(starts) if compute_window_cost_classical(grid_bits, start, N, M) == 0]


def gray_order_valid(W, IDX):
    """
    000 -> 0
    001 -> 1
    011 -> 3
    010 -> 2
    110 -> 6
    111 -> 7
    101 -> 5
    100 -> 4
    """
    gray_full = [t ^ (t >> 1) for t in range(2**IDX)]
    return [g for g in gray_full if g < W]


def format_nd_array_from_bits(bitstring, dims):
    arr = np.array(list(bitstring), dtype=str).reshape(tuple(dims))
    return np.array2string(arr, separator=' ').replace("'", "")

In [ ]:
# =========================================================
# Quantum blocks: reversible loader, delta cost phase, and mixers
# =========================================================

def append_mcx(qc, controls, target):
    """Modern MCX through MCXGate, without deprecated arguments such as mode='noancilla'."""
    if len(controls) == 0:
        qc.x(target)
    elif len(controls) == 1:
        qc.cx(controls[0], target)
    else:
        qc.append(MCXGate(num_ctrl_qubits=len(controls)), controls + [target])


def apply_window_loader(qc, n, idx, m, starts, N, M, order_valid):
    """
    Implements L: |grid>|i>|m> -> |grid>|i>|m xor window_i>.
    Applying this function twice uncomputes the window because the block is reversible XOR.
    """
    IDX = len(idx)
    current_zero_mask = [False] * IDX

    for i in order_valid:
        bits = [(i >> b) & 1 for b in range(IDX)]  # little-endian en Qiskit
        target_zero_mask = [bits[b] == 0 for b in range(IDX)]

        for b in range(IDX):
            if current_zero_mask[b] != target_zero_mask[b]:
                qc.x(idx[b])
                current_zero_mask[b] = target_zero_mask[b]

        for j, n_pos in enumerate(window_qubits_nd(starts[i], N, M)):
            controls = [idx[b] for b in range(IDX)] + [n[n_pos]]
            append_mcx(qc, controls, m[j])

    for b in range(IDX):
        if current_zero_mask[b]:
            qc.x(idx[b])


def apply_window_loader_shifted(
    qc, n, idx, m2, starts, N, M, order_valid,
    shift=1, boundary_mode="zero",
):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    """
    Loads window i+shift into m2 when the index register holds |i>.
    Boundary modes:
      zero: out-of-range shifted windows are left as |0>.
      same: out-of-range shifted windows reuse window i.
      wrap: shifted windows wrap modulo W.
    """
    IDX = len(idx)
    W = len(starts)
    current_zero_mask = [False] * IDX

    for i in order_valid:
        j = i + shift
        if j < W:
            source_index = j
        elif boundary_mode == "zero":
            source_index = None
        elif boundary_mode == "same":
            source_index = i
        elif boundary_mode == "wrap":
            source_index = j % W
        else:
            raise ValueError(f"Unknown boundary_mode: {boundary_mode}")

        bits = [(i >> b) & 1 for b in range(IDX)]
        target_zero_mask = [bits[b] == 0 for b in range(IDX)]

        for b in range(IDX):
            if current_zero_mask[b] != target_zero_mask[b]:
                qc.x(idx[b])
                current_zero_mask[b] = target_zero_mask[b]

        if source_index is None:
            continue

        for j_m, n_pos in enumerate(window_qubits_nd(starts[source_index], N, M)):
            controls = [idx[b] for b in range(IDX)] + [n[n_pos]]
            append_mcx(qc, controls, m2[j_m])

    for b in range(IDX):
        if current_zero_mask[b]:
            qc.x(idx[b])


def apply_cost_phase(qc, m, theta):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    """Compatibility wrapper retained for callers; V5 uses apply_delta_cost_phase_layer."""
    for q in m:
        qc.p(theta, q)


def _cost_adder_gate(n_m, n_a):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    cache = getattr(_cost_adder_gate, "_cache", {})
    key = (int(n_m), int(n_a))
    if key in cache:
        return cache[key]

    dim = 2 ** (n_m + n_a)
    perm = np.zeros((dim, dim), dtype=complex)
    m_mask = (1 << n_m) - 1
    for state in range(dim):
        m_bits = state & m_mask
        anc_val = state >> n_m
        cost = bin(m_bits).count("1")
        new_anc = (anc_val + cost) % (1 << n_a)
        new_state = (new_anc << n_m) | m_bits
        perm[new_state, state] = 1.0

    gate = UnitaryGate(perm, label="CostAdder")
    cache[key] = gate
    _cost_adder_gate._cache = cache
    return gate


def apply_cost_to_ancilla(qc, m, anc_cost):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    """
    Reversibly computes sum(m) into anc_cost.
    anc_cost must be initialised to |0> and have ceil(log2(len(m)+1)) qubits.
    """
    n_m = len(m)
    n_a = len(anc_cost)
    if n_m <= 10:
        qc.append(_cost_adder_gate(n_m, n_a), list(m) + list(anc_cost))
        return

    for control in m:
        for b in range(n_a - 1, 0, -1):
            append_mcx(qc, [control] + list(anc_cost[:b]), anc_cost[b])
        qc.cx(control, anc_cost[0])


def apply_cost_from_ancilla(qc, m, anc_cost):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    """Inverse of apply_cost_to_ancilla: subtracts sum(m) from anc_cost."""
    n_m = len(m)
    n_a = len(anc_cost)
    if n_m <= 10:
        gate = _cost_adder_gate(n_m, n_a).inverse()
        gate.label = "CostAdder†"
        qc.append(gate, list(m) + list(anc_cost))
        return

    for control in reversed(list(m)):
        qc.cx(control, anc_cost[0])
        for b in range(1, n_a):
            append_mcx(qc, [control] + list(anc_cost[:b]), anc_cost[b])


def _delta_phase_gate(n_a, theta):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    theta_value = float(theta)
    dim = 2 ** (2 * n_a)
    diag = np.ones(dim, dtype=complex)
    mask = (1 << n_a) - 1
    for state in range(dim):
        ca = state & mask
        cb = state >> n_a
        diag[state] = np.exp(1j * theta_value * (cb - ca))
    return UnitaryGate(np.diag(diag), label="DeltaPhase(theta)")


def apply_delta_cost_phase(qc, m, m2, anc, theta):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    """
    Applies phase exp(i·θ·ΔC(i)) where ΔC(i) = C(i+1) - C(i).
    The cost registers are uncomputed before returning.
    """
    M_tot = len(m)
    n_anc_bits = int(np.ceil(np.log2(M_tot + 1)))
    expected = 2 * n_anc_bits + 1
    if len(anc) != expected:
        raise ValueError(f"anc has {len(anc)} qubits, expected {expected}.")

    anc_a = anc[:n_anc_bits]
    anc_b = anc[n_anc_bits:2 * n_anc_bits]

    apply_cost_to_ancilla(qc, m, anc_a)
    apply_cost_to_ancilla(qc, m2, anc_b)
    qc.append(_delta_phase_gate(n_anc_bits, theta), list(anc_a) + list(anc_b))
    apply_cost_from_ancilla(qc, m2, anc_b)
    apply_cost_from_ancilla(qc, m, anc_a)


def _barrier(qc, enabled, label):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    if enabled:
        qc.barrier(label=label)


def apply_delta_cost_phase_layer(
    qc, n, idx, m, m2, anc, starts, N, M, order_valid, theta,
    shift=1, boundary_mode="zero", add_barriers=True, layer_label="",
):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    """Full V5 oracle layer: load i, load i+shift, phase Delta C, uncompute in reverse."""
    prefix = f"{layer_label}_" if layer_label else ""
    apply_window_loader(qc, n, idx, m, starts, N, M, order_valid)
    _barrier(qc, add_barriers, f"{prefix}load_i")
    apply_window_loader_shifted(
        qc, n, idx, m2, starts, N, M, order_valid,
        shift=shift, boundary_mode=boundary_mode,
    )
    _barrier(qc, add_barriers, f"{prefix}load_i_plus_1")
    apply_delta_cost_phase(qc, m, m2, anc, theta)
    _barrier(qc, add_barriers, f"{prefix}delta_phase")
    apply_window_loader_shifted(
        qc, n, idx, m2, starts, N, M, order_valid,
        shift=shift, boundary_mode=boundary_mode,
    )
    _barrier(qc, add_barriers, f"{prefix}unload_i_plus_1")
    apply_window_loader(qc, n, idx, m, starts, N, M, order_valid)
    _barrier(qc, add_barriers, f"{prefix}unload_i")


def two_level_mixer_gate(num_qubits, a, b, beta, label=None):
    """
    Local rotation in the subspace span{|a>, |b>}:
        exp(-i beta X_ab)
    and identity action on the rest. This is a modular prototype for small graphs.
    """
    dim = 2**num_qubits
    U = np.eye(dim, dtype=complex)
    c = np.cos(beta)
    s = -1j * np.sin(beta)
    U[a, a] = c
    U[b, b] = c
    U[a, b] = s
    U[b, a] = s
    return UnitaryGate(U, label=label or f"Mix({a},{b})")


def mixer_edges_from_starts(starts, N, method="local_geometric"):
    """Local edges between valid indices. Does not implement Grover global diffusion."""
    if method == "linear_valid":
        return [(i, i + 1) for i in range(len(starts) - 1)]

    if method != "local_geometric":
        raise ValueError("Unknown local mixer method.")

    start_to_idx = {tuple(s): i for i, s in enumerate(starts)}
    edges = []
    D = len(N)
    for i, start in enumerate(starts):
        for d in range(D):
            neigh = list(start)
            neigh[d] += 1
            neigh = tuple(neigh)
            if neigh in start_to_idx:
                edges.append((i, start_to_idx[neigh]))
    return edges


def apply_mixer(qc, idx, starts, N, mixer_angle, method="local_geometric"):
    """
    Modular mixer on the idx register.
    - local_geometric: mixes neighboring windows in the geometry of starts.
    - linear_valid: mixes consecutive valid indices.
    - rx_all: simple prototype; can transfer amplitude to invalid indices if W is not a power of 2.
    """
    if abs(mixer_angle) < 1e-15:
        return

    if method == "rx_all":
        for q in idx:
            qc.rx(2 * mixer_angle, q)
        return

    IDX = len(idx)
    for a, b in mixer_edges_from_starts(starts, N, method):
        qc.append(two_level_mixer_gate(IDX, a, b, mixer_angle), list(idx))


In [ ]:
# =========================================================
# Circuit construction and probability analysis
# =========================================================

def prepare_valid_index_superposition(qc, idx, W):
    IDX = len(idx)
    amps = np.zeros(2**IDX, dtype=complex)
    amps[:W] = 1 / np.sqrt(W)
    qc.initialize(amps, idx)


def prepare_grid_register(qc, n, N, occupied_coords):
    for q in [coord_to_index(normalize_coord(c, len(N)), N) for c in occupied_coords]:
        qc.x(n[q])


def compute_delta_costs_classical(costs, shift=1, boundary_mode="zero"):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    W = len(costs)
    delta_costs = []
    for i in range(W):
        j = i + shift
        if j < W:
            delta_costs.append(costs[j] - costs[i])
        elif boundary_mode == "zero":
            delta_costs.append(0 - costs[i])
        elif boundary_mode == "same":
            delta_costs.append(0)
        elif boundary_mode == "wrap":
            delta_costs.append(costs[j % W] - costs[i])
        else:
            raise ValueError(f"Unknown boundary_mode: {boundary_mode}")
    return delta_costs


def build_cost_phase_mixer_circuit(
    N, M, occupied_coords, theta, mixer_angle, repetitions,
    use_cost_phase=True, use_mixer=True, mixer_method="local_geometric",
    add_barriers=True, shift=1, boundary_mode="zero",
):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    validate_problem(N, M)
    D = len(N)
    N_tot = prod(N)
    M_tot = prod(M)
    starts = valid_starts_nd(N, M)
    W = len(starts)
    IDX = int(np.ceil(np.log2(W))) if W > 1 else 1
    order_valid = gray_order_valid(W, IDX)
    grid_bits = build_grid_bits(N, occupied_coords)
    costs = [compute_window_cost_classical(grid_bits, s, N, M) for s in starts]
    delta_costs = compute_delta_costs_classical(costs, shift=shift, boundary_mode=boundary_mode)
    n_anc_bits = int(np.ceil(np.log2(M_tot + 1)))
    n_anc = 2 * n_anc_bits + 1

    n = QuantumRegister(N_tot, "n")
    idx = QuantumRegister(IDX, "i")
    m = QuantumRegister(M_tot, "m")
    m2 = QuantumRegister(M_tot, "m2")
    anc = QuantumRegister(n_anc, "anc")
    qc = QuantumCircuit(n, idx, m, m2, anc)

    prepare_grid_register(qc, n, N, occupied_coords)
    prepare_valid_index_superposition(qc, idx, W)
    if add_barriers:
        qc.barrier(label="prepare")

    for r in range(repetitions):
        if use_cost_phase:
            apply_delta_cost_phase_layer(
                qc, n, idx, m, m2, anc, starts, N, M, order_valid, theta,
                shift=shift, boundary_mode=boundary_mode,
                add_barriers=add_barriers, layer_label=f"r{r}",
            )
        if use_mixer:
            apply_mixer(qc, idx, starts, N, mixer_angle, mixer_method)
        if add_barriers:
            qc.barrier(label="delta_cost_phase_mixer_layer")

    metadata = {
        "D": D, "N": list(N), "M": list(M), "N_tot": N_tot, "M_tot": M_tot,
        "starts": starts, "W": W, "IDX": IDX, "grid_bits": grid_bits,
        "occupied_coords": [normalize_coord(c, D) for c in occupied_coords],
        "theta": theta, "mixer_angle": mixer_angle, "repetitions": repetitions,
        "mixer_method": mixer_method,
        "costs": costs,
        "delta_costs": delta_costs,
        "oracle": "delta_cost",
        "shift": shift,
        "boundary_mode": boundary_mode,
        "n_anc": n_anc,
        "n_anc_bits": n_anc_bits,
        "m2_register": True,
    }
    return qc, metadata


def index_probabilities_from_statevector(sv, metadata):
    N_tot = metadata["N_tot"]
    IDX = metadata["IDX"]
    probs = np.zeros(2**IDX, dtype=float)
    idx_mask = (1 << IDX) - 1
    for basis_idx, amp in enumerate(sv.data):
        prob = float(abs(amp) ** 2)
        if prob == 0.0:
            continue
        idx_int = (basis_idx >> N_tot) & idx_mask
        probs[idx_int] += prob
    return probs


def analyze_probabilities(sv, metadata, title="Analysis", tol=1e-12):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    N, M = metadata["N"], metadata["M"]
    starts = metadata["starts"]
    W = metadata["W"]
    grid_bits = metadata["grid_bits"]
    probs = index_probabilities_from_statevector(sv, metadata)
    valid_indices = get_valid_indices(grid_bits, starts, N, M)
    delta_costs = metadata.get("delta_costs")
    if delta_costs is None:
        costs = [compute_window_cost_classical(grid_bits, s, N, M) for s in starts]
        delta_costs = compute_delta_costs_classical(costs)

    p_valid_initial = len(valid_indices) / W
    p_valid_after = float(sum(probs[i] for i in valid_indices))
    p_invalid_initial = 1.0 - p_valid_initial
    p_invalid_after = 1.0 - p_valid_after
    p_invalid_index = float(sum(probs[i] for i in range(W, len(probs))))

    print(f"\n============ {title} ============")
    print(f"N={N}, M={M}, W={W}, IDX={metadata['IDX']}")
    print(f"theta={metadata['theta']:.6g}, mixer_angle={metadata['mixer_angle']:.6g}, repetitions={metadata['repetitions']}")
    print(f"mixer_method={metadata['mixer_method']}, oracle={metadata.get('oracle', 'delta_cost')}")
    print(f"valid_indices={valid_indices}")
    print(f"P_valid_initial = {p_valid_initial:.6f}")
    print(f"P_valid_after   = {p_valid_after:.6f}")
    print(f"P_invalid_initial = {p_invalid_initial:.6f}")
    print(f"P_invalid_after = {p_invalid_after:.6f}")
    if p_invalid_index > tol:
        print(f"P_invalid_index_states = {p_invalid_index:.6f}")

    print("\nindex | start coordinate | window | ΔC(i) | probability | valid")
    print("------|------------------|--------|------------|-------------|------")
    rows = []
    for i, start in enumerate(starts):
        window = window_string_classical(grid_bits, start, N, M)
        cost = compute_window_cost_classical(grid_bits, start, N, M)
        delta_cost = delta_costs[i]
        rows.append((i, start, window, cost, delta_cost, probs[i]))
        print(f"{i:5d} | {str(start):16s} | {window:6s} | {delta_cost:10d} | {probs[i]:11.6f} | {cost == 0}")

    return {
        "probs": probs,
        "valid_indices": valid_indices,
        "P_valid_initial": p_valid_initial,
        "P_invalid_initial": p_invalid_initial,
        "P_valid_after": p_valid_after,
        "P_invalid_after": p_invalid_after,
        "P_invalid_index_states": p_invalid_index,
        "rows": rows,
    }


def run_experiment(name, N, M, occupied_coords, theta=np.pi, mixer_angle=0.35, repetitions=2, use_cost_phase=True,
                   use_mixer=True, mixer_method="local_geometric", draw=False, shift=1, boundary_mode="zero"):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    print(f"\n\n########################################")
    print(f"Experiment: {name}")
    print(f"########################################")
    qc, metadata = build_cost_phase_mixer_circuit(
        N=N, M=M, occupied_coords=occupied_coords,
        theta=theta, mixer_angle=mixer_angle, repetitions=repetitions,
        use_cost_phase=use_cost_phase, use_mixer=use_mixer,
        mixer_method=mixer_method,
        shift=shift, boundary_mode=boundary_mode,
    )
    sv = Statevector(qc)
    analysis = analyze_probabilities(sv, metadata, title=name)
    if draw:
        display(qc.draw(output="mpl"))
    return qc, metadata, analysis


## Small tests

These tests are not formal unit tests: they are small cases used to inspect whether the valid probability increases or decreases when changing `theta`, `mixer_angle`, `repetitions`, and `mixer_method`. If a case does not improve `P_valid`, that does not imply a circuit bug; it means that the selected parameters or mixer are not suitable for that instance.

In [ ]:
qc_test_1d, meta_test_1d, analysis_test_1d = run_experiment(
    name="Test 1D: consecutive occupied block",
    N=[8],
    M=[2],
    occupied_coords=[(3,), (4,)],
    theta=np.pi,
    mixer_angle=0.30,
    repetitions=4,
    draw=True
)

# qc_test_2d, meta_test_2d, analysis_test_2d = run_experiment(
#     name="Compact 2D test: 3x3 with 2x2 window",
#     N=[3, 3],
#     M=[2, 2],
#     occupied_coords=[(0, 0), (0, 2), (2, 2)],
#     theta=np.pi / 2,
#     mixer_angle=0.28,
#     repetitions=2,
#     draw=True
# )

In [ ]:
# =========================================================
# V5 phase profile analysis
# =========================================================

import matplotlib.pyplot as plt
from pathlib import Path

profile_meta = meta_test_1d
costs_profile = np.asarray(profile_meta["costs"], dtype=float)
delta_profile = np.asarray(profile_meta["delta_costs"], dtype=float)
theta_profile = float(profile_meta["theta"])
phase_profile = theta_profile * delta_profile

delta_colors = [
    "steelblue" if value < 0 else "darkorange" if value > 0 else "0.55"
    for value in delta_profile
]
valid_color = globals().get("VALID_COLOR", "#2ecc71")
invalid_color = globals().get("INVALID_COLOR", "#e74c3c")
cost_colors = [valid_color if value == 0 else invalid_color for value in costs_profile]
x = np.arange(profile_meta["W"])

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

axes[0].bar(x, costs_profile, color=cost_colors)
axes[0].set_xlabel("window index")
axes[0].set_ylabel("cost")
axes[0].set_title("Cost profile C(i)")

axes[1].bar(x, delta_profile, color=delta_colors)
axes[1].axhline(0, color="black", linewidth=1.0)
axes[1].set_xlabel("window index")
axes[1].set_ylabel("ΔC")
axes[1].set_title("Delta cost profile ΔC(i)")

axes[2].bar(x, phase_profile, color=delta_colors)
axes[2].axhline(0, color="black", linewidth=1.0)
axes[2].set_xlabel("window index")
axes[2].set_ylabel("phase")
axes[2].set_title("Phase profile φᵢ = θ·ΔC(i)")

fig.tight_layout()
phase_profile_dir = Path("TFG_V5_Analysis")
phase_profile_dir.mkdir(exist_ok=True)
fig.savefig(phase_profile_dir / "v5_phase_profile.pdf", bbox_inches="tight")
fig.savefig(phase_profile_dir / "v5_phase_profile.png", dpi=200, bbox_inches="tight")
plt.close(fig)


## Design notes

- The V5 phase is implemented by loading `window_i` into `m`, loading `window_{i+1}` into `m2`, computing both Hamming costs into ancilla registers, and applying a diagonal phase `exp(i·θ·ΔC(i))`.
- The cost ancilla, `m2`, and `m` are then uncomputed in strict reverse order, leaving only the phase on `idx`.
- The default mixer is local on the graph of valid windows. It does not use Grover's global diffusion operator.
- `rx_all` remains as a simple prototype, but it can transfer amplitude to invalid indices when `W` is not a power of 2.
- For large instances, the local mixer based on `UnitaryGate` should be replaced by a more efficient decomposition into native gates or by specialized geometric mixers.


## Case-study analysis

This section now uses a single final best-first hybrid optimizer. The optimizer searches theta, beta, and r jointly in the reduced W x W model, then builds the selected Qiskit circuit once for final validation when the full-register statevector is feasible.

In [ ]:
import csv
import re
import time
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

try:
    plt.style.use("seaborn-v0_8-whitegrid")
except Exception:
    pass

plt.rcParams.update({
    "axes.titlesize": 14,
    "axes.labelsize": 13,
    "xtick.labelsize": 11,
    "ytick.labelsize": 11,
    "legend.fontsize": 11,
})

VALID_COLOR = "#2ecc71"
INVALID_COLOR = "#e74c3c"
BASELINE_COLOR = "0.45"

V5_OUTPUT_DIR = Path("TFG_V5_Analysis")
V5_OUTPUT_DIR.mkdir(exist_ok=True)


In [ ]:
V5_CASE_STUDIES = [
    {
        "name": "01_1d_tiny_single_gap",
        "description": "Minimal 1D case with one valid window.",
        "N": [6], "M": [2],
        "occupied_coords": [(0,), (3,), (4,)],
        "theta": np.pi / 2, "beta": 0.30, "mixer_method": "local_geometric",
    },
    {
        "name": "02_1d_main_reference",
        "description": "Reference 1D instance used in the manual main experiment.",
        "N": [8], "M": [2],
        "occupied_coords": [(0,), (1,), (2,), (6,), (7,)],
        "theta": np.pi / 3, "beta": 0.60, "mixer_method": "local_geometric",
    },
    {
        "name": "03_1d_two_free_regions",
        "description": "1D grid with two occupied blocks and a central valid plateau.",
        "N": [10], "M": [3],
        "occupied_coords": [(0,), (1,), (7,), (8,), (9,)],
        "theta": np.pi / 3, "beta": 0.30, "mixer_method": "local_geometric",
    },
    {
        "name": "04_1d_clustered_medium",
        "description": "Medium 1D case with several clustered obstacles.",
        "N": [16], "M": [3],
        "occupied_coords": [(0,), (1,), (5,), (6,), (7,), (13,), (14,)],
        "theta": np.pi / 3, "beta": 0.24, "mixer_method": "local_geometric",
    },
    {
        "name": "05_1d_long_clustered_blocks",
        "description": "Longer 1D case with separated occupied blocks.",
        "N": [32], "M": [4],
        "occupied_coords": [(0,), (1,), (2,), (9,), (10,), (11,), (18,), (19,), (28,), (29,), (30,), (31,)],
        "theta": np.pi / 4, "beta": 0.18, "mixer_method": "local_geometric",
    },
    {
        "name": "06_2d_tiny_corner_block",
        "description": "Small 2D case with a single valid 2x2 region.",
        "N": [3, 3], "M": [2, 2],
        "occupied_coords": [(0, 0), (0, 2), (2, 2)],
        "theta": np.pi / 2, "beta": 0.28, "mixer_method": "local_geometric",
    },
    {
        "name": "07_2d_small_diagonal_block",
        "description": "4x4 grid with diagonal obstacles and one clear lower-left solution.",
        "N": [4, 4], "M": [2, 2],
        "occupied_coords": [(1, 1), (2, 2), (0, 3)],
        "theta": np.pi / 2, "beta": 0.24, "mixer_method": "local_geometric",
    },
    {
        "name": "08_2d_medium_clustered_obstacles",
        "description": "5x5 grid with two compact occupied clusters.",
        "N": [5, 5], "M": [2, 2],
        "occupied_coords": [(0, 0), (0, 1), (1, 0), (3, 3), (3, 4), (4, 3)],
        "theta": np.pi / 3, "beta": 0.22, "mixer_method": "local_geometric",
    },
    {
        "name": "09_2d_rectangular_window",
        "description": "6x6 grid with a non-square 3x2 window.",
        "N": [6, 6], "M": [3, 2],
        "occupied_coords": [(0, 0), (0, 1), (1, 0), (4, 4), (4, 5), (5, 4), (2, 3)],
        "theta": np.pi / 3, "beta": 0.18, "mixer_method": "local_geometric",
    },
    {
        "name": "10_3d_small_clustered_obstacles",
        "description": "Small 3D grid with two compact occupied clusters.",
        "N": [4, 4, 3], "M": [2, 2, 2],
        "occupied_coords": [(0, 0, 0), (0, 0, 1), (1, 0, 0), (3, 3, 2), (3, 2, 2), (2, 3, 2)],
        "theta": np.pi / 4, "beta": 0.16, "mixer_method": "local_geometric",
    },
]

## Shared analysis utilities
Helper functions and constants shared by all subsequent analysis cells.
These do not produce any output when executed — they only define functions.


In [ ]:
# =========================================================
# Shared reduced-model and final-circuit utilities
# =========================================================
# The case-study execution below uses only the final best-first hybrid
# optimizer. These helpers intentionally keep only the final hybrid analysis utilities.

def v5_slug(text):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    return re.sub(r"[^a-zA-Z0-9_]+", "_", text).strip("_").lower()


def final_hybrid_savefig(fig, stem, output_dir=None):
    out_dir = Path(output_dir) if output_dir is not None else FINAL_HYBRID_OUTPUT_DIR
    out_dir.mkdir(exist_ok=True)
    fig.savefig(out_dir / f"{stem}.pdf", bbox_inches="tight")
    fig.savefig(out_dir / f"{stem}.png", dpi=200, bbox_inches="tight")
    plt.close(fig)


def v5_delta_costs_from_costs(costs, shift=1, boundary_mode="zero"):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    return compute_delta_costs_classical(costs, shift=shift, boundary_mode=boundary_mode)


def v5_case_context(case):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    N_case = list(case["N"])
    M_case = list(case["M"])
    occupied_case = list(case["occupied_coords"])
    validate_problem(N_case, M_case)
    starts_case = valid_starts_nd(N_case, M_case)
    grid_bits_case = build_grid_bits(N_case, occupied_case)
    costs_case = [
        compute_window_cost_classical(grid_bits_case, s, N_case, M_case)
        for s in starts_case
    ]
    shift_case = int(case.get("shift", 1))
    boundary_mode_case = case.get("boundary_mode", "zero")
    delta_costs_case = v5_delta_costs_from_costs(
        costs_case, shift=shift_case, boundary_mode=boundary_mode_case,
    )
    valid_indices_case = [i for i, c in enumerate(costs_case) if c == 0]
    if not valid_indices_case:
        raise ValueError(f"Case {case['name']} has no valid windows; change occupied_coords.")
    W_case = len(starts_case)
    IDX_case = int(np.ceil(np.log2(W_case))) if W_case > 1 else 1
    return {
        "name": case["name"],
        "description": case.get("description", ""),
        "N": N_case,
        "M": M_case,
        "occupied_coords": occupied_case,
        "starts": starts_case,
        "grid_bits": grid_bits_case,
        "costs": costs_case,
        "delta_costs": delta_costs_case,
        "shift": shift_case,
        "boundary_mode": boundary_mode_case,
        "oracle": "delta_cost",
        "valid_indices": valid_indices_case,
        "W": W_case,
        "IDX": IDX_case,
        "P_uniform": len(valid_indices_case) / W_case,
        "theta_default": float(case.get("theta", globals().get("theta", np.pi / 3))),
        "beta_default": float(case.get("beta", globals().get("mixer_angle", 0.25))),
        "mixer_method": case.get("mixer_method", "local_geometric"),
    }


def v5_two_level_matrix(W, a, b, beta):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    U = np.eye(W, dtype=complex)
    c = np.cos(beta)
    s = -1j * np.sin(beta)
    U[a, a] = c
    U[b, b] = c
    U[a, b] = s
    U[b, a] = s
    return U


def v5_phase_vector(ctx, theta_value):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    return theta_value * np.asarray(ctx["delta_costs"], dtype=float)


def v5_layer_matrix(ctx, theta_value, beta_value):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    W_case = ctx["W"]
    costs = np.asarray(ctx["costs"], dtype=float)
    shift = int(ctx.get("shift", 1))
    boundary_mode = ctx.get("boundary_mode", "zero")

    delta_costs = np.asarray(
        v5_delta_costs_from_costs(costs, shift=shift, boundary_mode=boundary_mode),
        dtype=float,
    )
    D = np.diag(np.exp(1j * theta_value * delta_costs))

    U_mix = np.eye(W_case, dtype=complex)
    edges = mixer_edges_from_starts(
        ctx["starts"], ctx["N"], ctx.get("mixer_method", "linear_valid"),
    )
    for a, b in edges:
        if a < W_case and b < W_case:
            U_mix = v5_two_level_matrix(W_case, a, b, beta_value) @ U_mix
    return U_mix @ D


def v5_initial_state(ctx):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    return np.ones(ctx["W"], dtype=complex) / np.sqrt(ctx["W"])


def v5_probabilities(ctx, theta_value, beta_value, reps_value):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    U = v5_layer_matrix(ctx, theta_value, beta_value)
    psi = v5_initial_state(ctx)
    for _ in range(int(reps_value)):
        psi = U @ psi
    return np.abs(psi) ** 2


def v5_p_valid(ctx, probs):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    return float(np.sum(np.asarray(probs)[ctx["valid_indices"]]))


def final_hybrid_final_circuit_simulation(ctx, theta_opt, beta_opt, r_opt, force_statevector=False):
    # [V5] Modified/Added for ΔC(i) domain-wall approach
    """
    Optionally build and simulate the official V5 Qiskit circuit for the
    selected parameters. By default the case-study loop skips this expensive
    Statevector validation and returns the reduced-model final probabilities.
    Set FINAL_HYBRID_SIMULATE_QISKIT_CIRCUIT = True to enable it.
    """
    reduced_probs = v5_probabilities(ctx, theta_opt, beta_opt, r_opt)
    reduced_p_valid = v5_p_valid(ctx, reduced_probs)
    simulate_qiskit = bool(globals().get("FINAL_HYBRID_SIMULATE_QISKIT_CIRCUIT", False))

    if not simulate_qiskit and not force_statevector:
        return {
            "circuit": None,
            "metadata": None,
            "probs": reduced_probs,
            "p_valid": reduced_p_valid,
            "p_valid_reduced": reduced_p_valid,
            "qiskit_statevector_used": False,
            "qiskit_circuit_built": False,
            "circuit_depth": None,
            "circuit_gates": {},
            "num_qubits": None,
        }

    M_tot = prod(ctx["M"])
    n_anc_bits = int(np.ceil(np.log2(M_tot + 1)))
    n_anc = 2 * n_anc_bits + 1
    total_qubits = prod(ctx["N"]) + ctx["IDX"] + 2 * M_tot + n_anc
    if total_qubits > globals().get("FINAL_QISKIT_STATEVECTOR_QUBIT_LIMIT", 24) and not force_statevector:
        print(
            f"Warning: circuit has {total_qubits} qubits. Statevector simulation "
            "may be slow. Consider reducing grid size for testing."
        )
        return {
            "circuit": None,
            "metadata": None,
            "probs": reduced_probs,
            "p_valid": reduced_p_valid,
            "p_valid_reduced": reduced_p_valid,
            "qiskit_statevector_used": False,
            "qiskit_circuit_built": False,
            "circuit_depth": None,
            "circuit_gates": {},
            "num_qubits": total_qubits,
        }

    qc, metadata = build_cost_phase_mixer_circuit(
        ctx["N"], ctx["M"], ctx["occupied_coords"],
        theta_opt, beta_opt, r_opt,
        mixer_method=ctx.get("mixer_method", "local_geometric"),
        shift=ctx.get("shift", 1),
        boundary_mode=ctx.get("boundary_mode", "zero"),
    )
    circuit_depth = qc.depth()
    circuit_gates = dict(qc.count_ops())
    qubit_limit = globals().get("FINAL_QISKIT_STATEVECTOR_QUBIT_LIMIT", 24)

    if qc.num_qubits > qubit_limit and not force_statevector:
        print(
            f"Warning: circuit has {qc.num_qubits} qubits. Statevector simulation "
            "may be slow. Consider reducing grid size for testing."
        )
        return {
            "circuit": qc,
            "metadata": metadata,
            "probs": reduced_probs,
            "p_valid": reduced_p_valid,
            "p_valid_reduced": reduced_p_valid,
            "qiskit_statevector_used": False,
            "qiskit_circuit_built": True,
            "circuit_depth": circuit_depth,
            "circuit_gates": circuit_gates,
            "num_qubits": qc.num_qubits,
        }

    try:
        sv = Statevector(qc)
        idx_probs = index_probabilities_from_statevector(sv, metadata)
        p_valid_final = v5_p_valid(ctx, idx_probs)
        tolerance = globals().get("FINAL_HYBRID_MODEL_TOL", 1e-6)
        if abs(p_valid_final - reduced_p_valid) > tolerance:
            print(
                f"[Final hybrid warning] {ctx['name']}: reduced model P_valid "
                f"{reduced_p_valid:.6f} differs from Qiskit Statevector "
                f"{p_valid_final:.6f}."
            )
        return {
            "circuit": qc,
            "metadata": metadata,
            "probs": idx_probs,
            "p_valid": p_valid_final,
            "p_valid_reduced": reduced_p_valid,
            "qiskit_statevector_used": True,
            "qiskit_circuit_built": True,
            "circuit_depth": circuit_depth,
            "circuit_gates": circuit_gates,
            "num_qubits": qc.num_qubits,
        }
    except Exception as exc:
        print(
            f"[Final hybrid warning] {ctx['name']}: Statevector validation failed "
            f"({exc}). Returning reduced-model probabilities after building the "
            "full circuit once."
        )
        return {
            "circuit": qc,
            "metadata": metadata,
            "probs": reduced_probs,
            "p_valid": reduced_p_valid,
            "p_valid_reduced": reduced_p_valid,
            "qiskit_statevector_used": False,
            "qiskit_circuit_built": True,
            "circuit_depth": circuit_depth,
            "circuit_gates": circuit_gates,
            "num_qubits": qc.num_qubits,
        }

print("Final hybrid shared utilities loaded.")


In [ ]:
# =========================================================
# V5 uncomputation and reduced-model verification
# =========================================================

def verify_v5_uncomputation_for_first_case():
    first_case = V5_CASE_STUDIES[0]
    ctx = v5_case_context(first_case)
    theta = ctx["theta_default"]
    beta = ctx["beta_default"]

    qc, metadata = build_cost_phase_mixer_circuit(
        ctx["N"], ctx["M"], ctx["occupied_coords"],
        theta, beta, repetitions=1,
        mixer_method=ctx.get("mixer_method", "local_geometric"),
        shift=ctx.get("shift", 1),
        boundary_mode=ctx.get("boundary_mode", "zero"),
    )
    sv = Statevector(qc)

    N_tot = metadata["N_tot"]
    IDX = metadata["IDX"]
    M_tot = metadata["M_tot"]
    n_anc = metadata["n_anc"]
    m_shift = N_tot + IDX
    m2_shift = m_shift + M_tot
    anc_shift = m2_shift + M_tot
    m_mask = (1 << M_tot) - 1
    anc_mask = (1 << n_anc) - 1

    anc_population = 0.0
    m_residual = 0.0
    m2_residual = 0.0
    for basis_idx, amp in enumerate(sv.data):
        prob = float(abs(amp) ** 2)
        if prob == 0.0:
            continue
        if ((basis_idx >> anc_shift) & anc_mask) != 0:
            anc_population += prob
        if ((basis_idx >> m_shift) & m_mask) != 0:
            m_residual += prob
        if ((basis_idx >> m2_shift) & m_mask) != 0:
            m2_residual += prob

    if anc_population < 1e-6:
        print(f"Uncomputation OK: anc residual = {anc_population:.2e}")
    else:
        print(f"WARNING: anc not uncomputed, residual = {anc_population:.2e}")
    print(f"m uncomputed: {m_residual:.2e}, m2 uncomputed: {m2_residual:.2e}")

    idx_probs = index_probabilities_from_statevector(sv, metadata)
    p_valid_full = v5_p_valid(ctx, idx_probs)
    matrix_probs = v5_probabilities(ctx, theta, beta, 1)
    p_valid_matrix = v5_p_valid(ctx, matrix_probs)
    diff = abs(p_valid_full - p_valid_matrix)
    print(f"Full circuit P_valid: {p_valid_full:.8f}")
    print(f"Matrix method P_valid: {p_valid_matrix:.8f}")
    print(f"Matrix vs circuit agreement: delta = {diff:.2e}")


verify_v5_uncomputation_for_first_case()


## Final case-study analysis

Note: The spectrum plot shows the eigenphases of the reduced one-layer unitary `U(theta_opt, beta_opt)`. Repeated application of this layer produces phase accumulation `exp(i r phi_j)`. The interference between eigencomponents is what causes oscillations in `P_valid` as `r` changes. Therefore, the spectrum helps diagnose whether the chosen layer has a clean oscillatory structure and why certain `r` values amplify valid windows.


In [ ]:
# =========================================================
# Final best-first hybrid optimizer for all case studies
# =========================================================
# Case-study flow:
# 1. Coarse global search over theta and beta.
# 2. For every theta, beta pair, optimise r over 1..HYBRID_R_MAX.
# 3. Keep the top candidates and locally refine them with local grid search.
# 4. Re-optimise r whenever theta or beta changes.
# 5. Choose the hardware-aware candidate within HARDWARE_P_TOL of P_max.
# 6. Build the final Qiskit circuit once and validate with Statevector when feasible.

FINAL_HYBRID_OUTPUT_DIR = Path("TFG_V5_Analysis")
FINAL_HYBRID_OUTPUT_DIR.mkdir(exist_ok=True)

HYBRID_THETA_POINTS = 15
HYBRID_BETA_POINTS = 15
HYBRID_R_MAX = 100
HYBRID_TOP_K = 12
HYBRID_REFINE_ROUNDS = 3
HYBRID_REFINE_GRID_POINTS = 5
HARDWARE_P_TOL = 0.01
HARDWARE_LAMBDA = 0.002
FINAL_HYBRID_SIMULATE_QISKIT_CIRCUIT = False
FINAL_QISKIT_STATEVECTOR_QUBIT_LIMIT = 24
FINAL_HYBRID_MODEL_TOL = 1e-6


def candidate_hardware_score(candidate, hardware_lambda=HARDWARE_LAMBDA):
    return float(candidate["p_valid"] - hardware_lambda * int(candidate["r"]))


def evaluate_theta_beta_with_best_r(ctx, theta, beta, r_max):
    """
    Uses the reduced W x W layer matrix to evaluate all r=1..r_max.
    Returns theta, beta, the best r, the best P_valid, and the full curve.
    """
    theta = float(np.clip(theta, 0.0, np.pi))
    beta = float(np.clip(beta, 0.0, np.pi / 2))
    U = v5_layer_matrix(ctx, theta, beta)
    psi = v5_initial_state(ctx)
    p_curve = np.zeros(int(r_max), dtype=float)
    best_r = 1
    best_p = -np.inf

    for idx in range(int(r_max)):
        psi = U @ psi
        p_value = v5_p_valid(ctx, np.abs(psi) ** 2)
        p_curve[idx] = p_value
        if p_value > best_p:
            best_p = float(p_value)
            best_r = idx + 1

    return {
        "theta": theta,
        "beta": beta,
        "r": int(best_r),
        "best_r": int(best_r),
        "p_valid": float(best_p),
        "best_P_valid": float(best_p),
        "p_curve": p_curve,
    }


def coarse_global_search(ctx, theta_points, beta_points, r_max, top_k):
    """
    Sweeps theta,beta and keeps the top_k candidates by P_valid.
    The heatmap stores the best P_valid over r for each theta,beta cell.
    """
    theta_values = np.linspace(0.0, np.pi, int(theta_points))
    beta_values = np.linspace(0.0, np.pi / 2, int(beta_points))
    heatmap_best_p = np.zeros((len(theta_values), len(beta_values)), dtype=float)
    heatmap_best_r = np.zeros((len(theta_values), len(beta_values)), dtype=int)
    candidates = []

    for i, theta_value in enumerate(theta_values):
        for j, beta_value in enumerate(beta_values):
            candidate = evaluate_theta_beta_with_best_r(ctx, theta_value, beta_value, r_max)
            heatmap_best_p[i, j] = candidate["p_valid"]
            heatmap_best_r[i, j] = candidate["r"]
            candidates.append(candidate)

    candidates.sort(key=lambda row: (row["p_valid"], -row["r"]), reverse=True)
    return {
        "candidates": candidates[: int(top_k)],
        "all_candidates": candidates,
        "heatmap": heatmap_best_p,
        "heatmap_r": heatmap_best_r,
        "theta_values": theta_values,
        "beta_values": beta_values,
    }


def deduplicate_candidates(candidates, ndigits=12):
    unique = {}
    for candidate in candidates:
        key = (
            round(float(candidate["theta"]), ndigits),
            round(float(candidate["beta"]), ndigits),
            int(candidate["r"]),
        )
        if key not in unique or candidate["p_valid"] > unique[key]["p_valid"]:
            unique[key] = candidate
    return list(unique.values())


def local_refinement(ctx, candidates, r_max, rounds, initial_radius_theta,
                     initial_radius_beta, grid_points):
    """
    Refines around the best candidates using local grid search.
    Every tested theta,beta recomputes the best r using local grid search only.
    The accepted local center never decreases P_valid.
    """
    expanded = []
    seeds = sorted(candidates, key=lambda row: row["p_valid"], reverse=True)

    for seed in seeds:
        current = seed
        radius_theta = float(initial_radius_theta)
        radius_beta = float(initial_radius_beta)
        expanded.append(seed)

        for _round in range(int(rounds)):
            theta_offsets = np.linspace(-radius_theta, radius_theta, int(grid_points))
            beta_offsets = np.linspace(-radius_beta, radius_beta, int(grid_points))
            local_candidates = []

            for dtheta in theta_offsets:
                theta_test = float(np.clip(current["theta"] + dtheta, 0.0, np.pi))
                for dbeta in beta_offsets:
                    beta_test = float(np.clip(current["beta"] + dbeta, 0.0, np.pi / 2))
                    local_candidates.append(
                        evaluate_theta_beta_with_best_r(ctx, theta_test, beta_test, r_max)
                    )

            local_candidates = deduplicate_candidates(local_candidates)
            local_candidates.sort(
                key=lambda row: (row["p_valid"], candidate_hardware_score(row), -row["r"]),
                reverse=True,
            )
            expanded.extend(local_candidates)
            best_local = local_candidates[0]

            current_score = candidate_hardware_score(current)
            best_score = candidate_hardware_score(best_local)
            improves_probability = best_local["p_valid"] > current["p_valid"] + 1e-12
            keeps_probability_and_scores_better = (
                abs(best_local["p_valid"] - current["p_valid"]) <= 1e-12
                and best_score > current_score
            )
            if improves_probability or keeps_probability_and_scores_better:
                current = best_local

            radius_theta *= 0.5
            radius_beta *= 0.5

    expanded = deduplicate_candidates(expanded)
    expanded.sort(
        key=lambda row: (row["p_valid"], candidate_hardware_score(row), -row["r"]),
        reverse=True,
    )
    return expanded


def select_hardware_aware_candidate(candidates, p_tol, hardware_lambda):
    """
    Selects the candidate with best score = P_valid - hardware_lambda*r
    among candidates within p_tol of the maximum P_valid.
    """
    if not candidates:
        raise ValueError("No candidates available for hardware-aware selection.")
    p_max = max(float(candidate["p_valid"]) for candidate in candidates)
    eligible = [
        dict(candidate)
        for candidate in candidates
        if float(candidate["p_valid"]) >= p_max - float(p_tol)
    ]
    for candidate in eligible:
        candidate["score"] = candidate_hardware_score(candidate, hardware_lambda)
        candidate["P_max"] = p_max
    eligible.sort(key=lambda row: (row["score"], row["p_valid"], -row["r"]), reverse=True)
    return eligible[0]


def best_first_hybrid_optimizer(ctx, theta_points=HYBRID_THETA_POINTS,
                                beta_points=HYBRID_BETA_POINTS,
                                r_max=HYBRID_R_MAX,
                                top_k=HYBRID_TOP_K,
                                refine_rounds=HYBRID_REFINE_ROUNDS,
                                refine_grid_points=HYBRID_REFINE_GRID_POINTS,
                                p_tol=HARDWARE_P_TOL,
                                hardware_lambda=HARDWARE_LAMBDA):
    """
    Final optimiser used for all case studies.
    Searches theta, beta, and r jointly using a best-first hybrid strategy.
    """
    coarse = coarse_global_search(ctx, theta_points, beta_points, r_max, top_k)
    initial_radius_theta = np.pi / max(1, int(theta_points) - 1)
    initial_radius_beta = (np.pi / 2) / max(1, int(beta_points) - 1)
    refined = local_refinement(
        ctx,
        coarse["candidates"],
        r_max=r_max,
        rounds=refine_rounds,
        initial_radius_theta=initial_radius_theta,
        initial_radius_beta=initial_radius_beta,
        grid_points=refine_grid_points,
    )
    all_candidates = deduplicate_candidates(coarse["all_candidates"] + refined)
    selected = select_hardware_aware_candidate(all_candidates, p_tol, hardware_lambda)
    selected = evaluate_theta_beta_with_best_r(ctx, selected["theta"], selected["beta"], r_max)
    selected["score"] = candidate_hardware_score(selected, hardware_lambda)
    selected["P_max"] = max(candidate["p_valid"] for candidate in all_candidates)
    return {
        "selected": selected,
        "coarse": coarse,
        "refined_candidates": refined,
        "all_candidates": all_candidates,
    }


def final_hybrid_spectrum_data(ctx, theta_opt, beta_opt):
    U = v5_layer_matrix(ctx, theta_opt, beta_opt)
    eigvals = np.linalg.eigvals(U)
    phases = np.sort(np.angle(eigvals))
    wrapped = np.r_[phases, phases[0] + 2 * np.pi]
    gaps = np.diff(wrapped)
    return {
        "eigvals": eigvals,
        "phases": phases,
        "gaps": gaps,
    }


def run_best_first_hybrid_case(ctx):
    """
    Runs the full final optimiser for one case study.
    The reduced model performs the search; the selected final circuit is
    built once with Qiskit and validated by Statevector when feasible.
    """
    optimizer_result = best_first_hybrid_optimizer(ctx)
    selected = optimizer_result["selected"]
    theta_opt = selected["theta"]
    beta_opt = selected["beta"]
    r_opt = selected["r"]
    final = final_hybrid_final_circuit_simulation(ctx, theta_opt, beta_opt, r_opt)
    spectrum = final_hybrid_spectrum_data(ctx, theta_opt, beta_opt)

    return {
        "case": ctx["name"],
        "ctx": ctx,
        "theta_opt": theta_opt,
        "beta_opt": beta_opt,
        "r_opt": r_opt,
        "r_star": r_opt,
        "P_valid": final["p_valid"],
        "P_valid_reduced": final["p_valid_reduced"],
        "P_uniform": ctx["P_uniform"],
        "heatmap": optimizer_result["coarse"]["heatmap"],
        "heatmap_r": optimizer_result["coarse"]["heatmap_r"],
        "theta_values": optimizer_result["coarse"]["theta_values"],
        "beta_values": optimizer_result["coarse"]["beta_values"],
        "oscillation_curve": selected["p_curve"],
        "spectrum": spectrum,
        "final_probs": final["probs"],
        "circuit_depth": final["circuit_depth"],
        "circuit_gates": final["circuit_gates"],
        "num_qubits": final["num_qubits"],
        "qiskit_statevector_used": final["qiskit_statevector_used"],
        "qiskit_circuit_built": final["qiskit_circuit_built"],
        "hardware_score": selected["score"],
        "P_max_reduced_search": selected["P_max"],
        "optimizer_result": optimizer_result,
    }


def draw_oscillation_plot(result, ax):
    reps = np.arange(1, len(result["oscillation_curve"]) + 1)
    ax.plot(reps, result["oscillation_curve"], marker="o", linewidth=2,
            color=VALID_COLOR, label="P_valid")
    ax.axvline(result["r_opt"], color="black", linestyle=":", linewidth=1.5,
               label=f"r_opt={result['r_opt']}")
    ax.axhline(result["P_uniform"], color=BASELINE_COLOR, linestyle="--",
               linewidth=1.5, label="uniform K/W")
    ax.set_xlabel("repetitions r")
    ax.set_ylabel("P_valid")
    ax.set_title(
        f"Oscillation - {result['case']}\n"
        f"theta={result['theta_opt']/np.pi:.3f}pi, "
        f"beta={result['beta_opt']/np.pi:.3f}pi, "
        f"r={result['r_opt']}, P={result['P_valid']:.4f}"
    )
    ax.legend(fontsize=9)


def draw_heatmap_plot(result, ax):
    theta_values = result["theta_values"]
    beta_values = result["beta_values"]
    im = ax.imshow(
        result["heatmap"],
        origin="lower",
        aspect="auto",
        cmap="viridis",
        extent=[
            beta_values[0] / np.pi,
            beta_values[-1] / np.pi,
            theta_values[0] / np.pi,
            theta_values[-1] / np.pi,
        ],
    )
    ax.scatter([result["beta_opt"] / np.pi], [result["theta_opt"] / np.pi],
               marker="*", s=180, color="white", edgecolor="black",
               linewidth=0.8, label="selected")
    ax.annotate(
        f"r={result['r_opt']}",
        xy=(result["beta_opt"] / np.pi, result["theta_opt"] / np.pi),
        xytext=(6, 6),
        textcoords="offset points",
        color="white",
        fontsize=9,
        weight="bold",
    )
    ax.set_xlabel("beta / pi")
    ax.set_ylabel("theta / pi")
    ax.set_title(f"Best-over-r heatmap - {result['case']}")
    ax.legend(loc="lower right", fontsize=9)
    return im


def draw_spectrum_plot(result, ax):
    # The eigenphases of U(theta_opt, beta_opt) drive the interference pattern
    # that appears in the repetition oscillation curve.
    eigvals = result["spectrum"]["eigvals"]
    phases = np.angle(eigvals) / np.pi
    unit = np.exp(1j * np.linspace(0, 2 * np.pi, 400))
    ax.plot(unit.real, unit.imag, color="0.75", linewidth=1.2)
    sc = ax.scatter(eigvals.real, eigvals.imag, c=phases, cmap="twilight",
                    s=55, edgecolor="black", linewidth=0.3)
    ax.set_aspect("equal", adjustable="box")
    ax.set_xlabel("Re(lambda)")
    ax.set_ylabel("Im(lambda)")
    ax.set_title(f"One-layer spectrum - {result['case']}")
    return sc


def draw_distribution_plot(result, ax):
    ctx = result["ctx"]
    probs = result["final_probs"]
    bar_colors = [
        VALID_COLOR if i in ctx["valid_indices"] else INVALID_COLOR
        for i in range(ctx["W"])
    ]
    ax.bar(np.arange(ctx["W"]), probs[: ctx["W"]], color=bar_colors)
    ax.axhline(1 / ctx["W"], color=BASELINE_COLOR, linestyle="--",
               linewidth=1.5, label="uniform 1/W")
    ax.set_xlabel("window index")
    ax.set_ylabel("probability")
    ax.set_title(
        f"Optimal distribution - {result['case']}\n"
        f"P={result['P_valid']:.4f}, theta={result['theta_opt']/np.pi:.3f}pi, "
        f"beta={result['beta_opt']/np.pi:.3f}pi, r={result['r_opt']}"
    )
    ax.legend(fontsize=9)


def save_final_hybrid_case_figures(result):
    slug = v5_slug(result["case"])

    fig, ax = plt.subplots(figsize=(7, 4.5))
    draw_oscillation_plot(result, ax)
    fig.tight_layout()
    final_hybrid_savefig(fig, f"{slug}_oscillation")

    fig, ax = plt.subplots(figsize=(7, 4.8))
    im = draw_heatmap_plot(result, ax)
    fig.colorbar(im, ax=ax, label="best P_valid over r")
    fig.tight_layout()
    final_hybrid_savefig(fig, f"{slug}_heatmap")

    fig, ax = plt.subplots(figsize=(6, 5.2))
    sc = draw_spectrum_plot(result, ax)
    fig.colorbar(sc, ax=ax, label="eigenphase / pi")
    fig.tight_layout()
    final_hybrid_savefig(fig, f"{slug}_spectrum")

    fig, ax = plt.subplots(figsize=(8, 4.5))
    draw_distribution_plot(result, ax)
    fig.tight_layout()
    final_hybrid_savefig(fig, f"{slug}_optimal_distribution")

    fig, axes = plt.subplots(2, 2, figsize=(13, 9.5))
    draw_oscillation_plot(result, axes[0, 0])
    im = draw_heatmap_plot(result, axes[0, 1])
    fig.colorbar(im, ax=axes[0, 1], label="best P_valid over r")
    sc = draw_spectrum_plot(result, axes[1, 0])
    fig.colorbar(sc, ax=axes[1, 0], label="eigenphase / pi")
    draw_distribution_plot(result, axes[1, 1])
    fig.suptitle(f"Final best-first hybrid summary - {result['case']}", fontsize=16)
    fig.tight_layout(rect=[0, 0.02, 1, 0.96])
    final_hybrid_savefig(fig, f"{slug}_summary")


def save_final_hybrid_cross_case_comparison(rows):
    names = [row["case"] for row in rows]
    x = np.arange(len(names))
    fig, ax = plt.subplots(figsize=(13, 5))
    ax.bar(x - 0.2, [row["P_uniform"] for row in rows], width=0.4,
           color="0.7", label="Uniform K/W")
    ax.bar(x + 0.2, [row["P_valid"] for row in rows], width=0.4,
           color=VALID_COLOR, label="Final optimal P_valid")
    for xi, row in zip(x, rows):
        ax.text(xi + 0.2, min(1.04, row["P_valid"] + 0.015),
                f"{row['P_valid']:.3f}", ha="center", va="bottom", fontsize=9)
    ax.set_xticks(x)
    ax.set_xticklabels([name.replace("_", "\n", 2) for name in names],
                       rotation=40, ha="right")
    ax.set_ylabel("P_valid")
    ax.set_ylim(0, 1.08)
    ax.set_title("Final hybrid: Uniform K/W vs optimal P_valid")
    ax.legend()
    fig.tight_layout()
    final_hybrid_savefig(fig, "final_hybrid_cross_case_comparison")


def write_dict_rows_csv(path, rows):
    if not rows:
        return
    fieldnames = []
    for row in rows:
        for key in row:
            if key not in fieldnames:
                fieldnames.append(key)
    with open(path, "w", newline="") as handle:
        writer = csv.DictWriter(handle, fieldnames=fieldnames)
        writer.writeheader()
        writer.writerows(rows)


FINAL_HYBRID_CASES = [v5_case_context(case) for case in V5_CASE_STUDIES]
final_hybrid_results = []
final_hybrid_rows = []
final_hybrid_curve_rows = []
final_hybrid_distribution_rows = []

for ctx in FINAL_HYBRID_CASES:
    print(
        f"\n[Final hybrid] {ctx['name']} | W={ctx['W']} | "
        f"K={len(ctx['valid_indices'])} | uniform={ctx['P_uniform']:.4f}"
    )
    result = run_best_first_hybrid_case(ctx)
    final_hybrid_results.append(result)
    save_final_hybrid_case_figures(result)

    row = {
        "case": result["case"],
        "oracle": "delta_cost",
        "shift": ctx.get("shift", 1),
        "boundary_mode": ctx.get("boundary_mode", "zero"),
        "n_anc_bits": int(np.ceil(np.log2(prod(ctx["M"]) + 1))),
        "n_anc": 2 * int(np.ceil(np.log2(prod(ctx["M"]) + 1))) + 1,
        "estimated_total_qubits": prod(ctx["N"]) + ctx["IDX"] + 2 * prod(ctx["M"]) + 2 * int(np.ceil(np.log2(prod(ctx["M"]) + 1))) + 1,
        "num_qubits": result["num_qubits"],
        "W": ctx["W"],
        "K": len(ctx["valid_indices"]),
        "P_uniform": result["P_uniform"],
        "theta_opt": result["theta_opt"],
        "beta_opt": result["beta_opt"],
        "theta_over_pi": result["theta_opt"] / np.pi,
        "beta_over_pi": result["beta_opt"] / np.pi,
        "r_opt": result["r_opt"],
        "r_star": result["r_opt"],
        "P_valid": result["P_valid"],
        "circuit_depth": result["circuit_depth"],
        "qiskit_statevector_used": result["qiskit_statevector_used"],
        "qiskit_circuit_built": result["qiskit_circuit_built"],
    }
    final_hybrid_rows.append(row)

    selected_curve = result["optimizer_result"]["selected"]["p_curve"]
    for rep_idx, p_value in enumerate(selected_curve, start=1):
        final_hybrid_curve_rows.append({
            "version": "V5",
            "case": result["case"],
            "theta_opt": result["theta_opt"],
            "beta_opt": result["beta_opt"],
            "theta_over_pi": result["theta_opt"] / np.pi,
            "beta_over_pi": result["beta_opt"] / np.pi,
            "r": rep_idx,
            "P_valid": float(p_value),
            "is_selected_r": int(rep_idx == result["r_opt"]),
        })

    final_probs = np.asarray(result["final_probs"], dtype=float)
    for window_idx, start in enumerate(ctx["starts"]):
        final_hybrid_distribution_rows.append({
            "version": "V5",
            "case": result["case"],
            "window_index": window_idx,
            "start": str(tuple(start)),
            "valid": window_idx in ctx["valid_indices"],
            "cost": ctx["costs"][window_idx],
            "delta_cost": ctx["delta_costs"][window_idx],
            "probability": float(final_probs[window_idx]),
        })

    print(f"  case name:     {row['case']}")
    print(f"  W:             {row['W']}")
    print(f"  K:             {row['K']}")
    print(f"  Uniform K/W:   {row['P_uniform']:.6f}")
    print(f"  theta_opt/pi:  {row['theta_over_pi']:.6f}")
    print(f"  beta_opt/pi:   {row['beta_over_pi']:.6f}")
    print(f"  r_opt:         {row['r_opt']}")
    print(f"  P_valid:       {row['P_valid']:.6f}")
    depth_label = row["circuit_depth"] if row["circuit_depth"] is not None else "skipped"
    print(f"  circuit depth: {depth_label}")
    write_dict_rows_csv(FINAL_HYBRID_OUTPUT_DIR / "v5_final_hybrid_results.csv", final_hybrid_rows)
    write_dict_rows_csv(FINAL_HYBRID_OUTPUT_DIR / "v5_selected_repetition_curves.csv", final_hybrid_curve_rows)
    write_dict_rows_csv(FINAL_HYBRID_OUTPUT_DIR / "v5_final_window_distribution.csv", final_hybrid_distribution_rows)

if final_hybrid_rows:
    print("\nFinal best-first hybrid summary")
    print("case | W | K | Uniform K/W | theta/pi | beta/pi | r_opt | P_valid | depth")
    print("-----|---|---|-------------|----------|---------|-------|---------|------")
    for row in final_hybrid_rows:
        print(
            f"{row['case']} | {row['W']} | {row['K']} | "
            f"{row['P_uniform']:.4f} | {row['theta_over_pi']:.4f} | "
            f"{row['beta_over_pi']:.4f} | {row['r_opt']} | "
            f"{row['P_valid']:.4f} | "
            f"{row['circuit_depth'] if row['circuit_depth'] is not None else 'skipped'}"
        )
    save_final_hybrid_cross_case_comparison(final_hybrid_rows)
    write_dict_rows_csv(FINAL_HYBRID_OUTPUT_DIR / "v5_final_hybrid_results.csv", final_hybrid_rows)
    write_dict_rows_csv(FINAL_HYBRID_OUTPUT_DIR / "v5_selected_repetition_curves.csv", final_hybrid_curve_rows)
    write_dict_rows_csv(FINAL_HYBRID_OUTPUT_DIR / "v5_final_window_distribution.csv", final_hybrid_distribution_rows)
    print(f"CSV database files saved in: {FINAL_HYBRID_OUTPUT_DIR}")
    print(f"\nFinal hybrid figures saved in: {FINAL_HYBRID_OUTPUT_DIR}")
else:
    print("No final hybrid case studies completed.")


## IBM quantum hardware execution

Executes the selected final hybrid circuit on real IBM quantum hardware. Parameters are taken from `final_hybrid_rows`; no re-optimisation is done here. Set `IBM_SUBMIT = True` to actually submit jobs.

In [ ]:
# =========================================================
# IBM quantum hardware execution
# =========================================================
# Uses the selected parameters from final_hybrid_rows.
# All optimisation is done classically; this cell only submits
# and retrieves the final circuit.
# =========================================================

IBM_SUBMIT          = False   # set True to actually submit
IBM_BACKEND_NAME    = "ibm_basquequantum"
IBM_SHOTS           = 4000
IBM_OPTIMIZATION_LEVEL = 3

def run_on_ibm(ctx, theta_opt, beta_opt, r_star,
               backend_name=IBM_BACKEND_NAME,
               shots=IBM_SHOTS,
               optimization_level=IBM_OPTIMIZATION_LEVEL):
    """
    Transpiles and submits the optimal V5 circuit to IBM hardware.
    Returns the job object and a result dict with estimated P_valid.
    """
    try:
        from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2
        from qiskit.transpiler.preset_passmanagers import (
            generate_preset_pass_manager,
        )
    except ImportError:
        print("qiskit_ibm_runtime not installed. "
              "Run: pip install qiskit-ibm-runtime")
        return None, None

    qc, _metadata = build_cost_phase_mixer_circuit(
        ctx["N"], ctx["M"], ctx["occupied_coords"],
        theta_opt, beta_opt, r_star,
        mixer_method=ctx.get(mixer_method, linear_valid),
        shift=ctx.get(shift, 1),
        boundary_mode=ctx.get(boundary_mode, zero),
    )

    # Add measurements on idx register only
    W    = ctx["W"]
    IDX  = ctx.get("IDX", int(np.ceil(np.log2(W))) if W > 1 else 1)
    N_tot = int(np.prod(ctx["N"]))
    from qiskit import ClassicalRegister
    cr = ClassicalRegister(IDX, "meas")
    qc.add_register(cr)
    idx_qubits = list(range(N_tot, N_tot + IDX))
    qc.measure(idx_qubits, list(range(IDX)))

    service = QiskitRuntimeService()
    backend = service.backend(backend_name)

    pm = generate_preset_pass_manager(
        optimization_level=optimization_level,
        backend=backend,
    )
    qc_transpiled = pm.run(qc)

    print(f"  Original depth:    {qc.depth()}")
    print(f"  Transpiled depth:  {qc_transpiled.depth()}")
    print(f"  Transpiled gates:  {qc_transpiled.count_ops()}")

    sampler = SamplerV2(mode=backend)
    job = sampler.run([qc_transpiled], shots=shots)
    print(f"  Job submitted: {job.job_id()}")
    return job, {"circuit": qc, "transpiled": qc_transpiled}


def interpret_ibm_result(job, ctx, theta_opt, beta_opt, r_star,
                         p_valid_ideal):
    """
    Retrieves counts from a finished IBM job and computes estimated
    P_valid. Compares with the ideal simulation result.
    """
    W   = ctx["W"]
    IDX = ctx.get("IDX", int(np.ceil(np.log2(W))) if W > 1 else 1)

    result = job.result()
    pub_result = result[0]
    counts_array = pub_result.data.meas.get_counts()

    valid_shots = 0
    total_shots = 0
    per_index   = np.zeros(W)

    for bitstring, count in counts_array.items():
        # Qiskit bitstrings are big-endian; reverse for little-endian index
        idx_int = int(bitstring[::-1], 2)
        total_shots += count
        if idx_int < W:
            per_index[idx_int] += count
            if idx_int in ctx["valid_indices"]:
                valid_shots += count

    p_valid_hardware = valid_shots / total_shots if total_shots > 0 else 0.0

    print(f"\n  IBM hardware result:")
    print(f"    Total shots:    {total_shots}")
    print(f"    Valid shots:    {valid_shots}")
    print(f"    P_valid (HW):   {p_valid_hardware:.4f}")
    print(f"    P_valid (sim):  {p_valid_ideal:.4f}")
    print(f"    Noise penalty:  {p_valid_ideal - p_valid_hardware:+.4f}")

    # Plot hardware vs ideal distribution
    bar_colors = [VALID_COLOR if i in ctx["valid_indices"]
                  else INVALID_COLOR for i in range(W)]
    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    ax = axes[0]
    ax.bar(np.arange(W), per_index[:W] / total_shots, color=bar_colors,
           alpha=0.9, label="hardware")
    ax.axhline(1 / W, color=BASELINE_COLOR, linestyle="--",
               linewidth=1.5, label="uniform 1/W")
    ax.set_xlabel("window index", fontsize=13)
    ax.set_ylabel("frequency", fontsize=13)
    ax.set_title(
        f"Hardware distribution — {ctx['name']}\n"
        f"P_valid={p_valid_hardware:.4f}, shots={total_shots}",
        fontsize=13
    )
    ax.legend(fontsize=11)

    ax = axes[1]
    ax.bar(np.arange(W), per_index[:W] / total_shots,
           color="steelblue", alpha=0.6, label="hardware")
    s4_probs = final_hybrid_final_circuit_simulation(ctx, theta_opt, beta_opt, r_star)["probs"]
    ax.bar(np.arange(W), s4_probs[:W],
           color=bar_colors, alpha=0.5, label="ideal simulation")
    ax.axhline(1 / W, color=BASELINE_COLOR, linestyle="--",
               linewidth=1.5, label="uniform 1/W")
    ax.set_xlabel("window index", fontsize=13)
    ax.set_ylabel("probability / frequency", fontsize=13)
    ax.set_title("Hardware vs ideal simulation", fontsize=13)
    ax.legend(fontsize=11)

    fig.tight_layout()
    for ext in ("pdf", "png"):
        fig.savefig(FINAL_HYBRID_OUTPUT_DIR / f"ibm_{v5_slug(ctx['name'])}.{ext}",
                    bbox_inches="tight",
                    dpi=200 if ext == "png" else None)
    plt.close(fig)

    return {
        "p_valid_hardware": p_valid_hardware,
        "p_valid_ideal":    p_valid_ideal,
        "noise_penalty":    p_valid_ideal - p_valid_hardware,
        "total_shots":      total_shots,
    }


# ----------------------------------------------------------
# Submit or retrieve jobs
# ----------------------------------------------------------
if IBM_SUBMIT and final_hybrid_rows:
    # Use the first final hybrid case as the IBM demo case.
    # Change the index to submit a different case.
    IBM_CASE_INDEX = 0
    demo_case = FINAL_HYBRID_CASES[IBM_CASE_INDEX]
    demo_row  = final_hybrid_rows[IBM_CASE_INDEX]

    print(f"Submitting IBM job for case: {demo_case['name']}")
    print(f"  theta={demo_row['theta_opt']/np.pi:.3f}π, "
          f"beta={demo_row['beta_opt']/np.pi:.3f}π, "
          f"r*={demo_row['r_star']}")

    ibm_job, ibm_meta = run_on_ibm(
        demo_case,
        demo_row["theta_opt"],
        demo_row["beta_opt"],
        demo_row["r_star"],
    )

    if ibm_job is not None:
        # Save job ID for later retrieval
        job_id_file = FINAL_HYBRID_OUTPUT_DIR / "ibm_job_id.txt"
        job_id_file.write_text(ibm_job.job_id())
        print(f"  Job ID saved to {job_id_file}")

elif not IBM_SUBMIT:
    print("IBM_SUBMIT = False. Set to True to submit jobs.")
    print("To retrieve a previously submitted job, set IBM_SUBMIT = False")
    print("and run the retrieval block below.")


In [ ]:
# ----------------------------------------------------------
# Retrieve and interpret a previously submitted job
# ----------------------------------------------------------
IBM_RETRIEVE_JOB_ID = None  # set to a job ID string to retrieve

if IBM_RETRIEVE_JOB_ID is not None and final_hybrid_rows:
    try:
        from qiskit_ibm_runtime import QiskitRuntimeService
        service  = QiskitRuntimeService()
        ibm_job  = service.job(IBM_RETRIEVE_JOB_ID)
        demo_row = final_hybrid_rows[0]
        demo_case = FINAL_HYBRID_CASES[0]

        ibm_result = interpret_ibm_result(
            ibm_job,
            demo_case,
            demo_row["theta_opt"],
            demo_row["beta_opt"],
            demo_row["r_star"],
            p_valid_ideal=demo_row["P_valid"],
        )
    except Exception as exc:
        print(f"Could not retrieve job: {exc}")
